# Step 3 - From a model alone to a complete RAG

We start with a language model and nothing else, and we add one piece at a time:

1. the model alone, on a general question;
2. the model alone, on a question about our documents;
3. the model with the right text pasted by hand;
4. the text found automatically: **retrieve**;
5. the text placed in the prompt with its source: **augment**;
6. the answer written from that text: **generate**.

At the end the three steps fit in one function: the complete RAG.

Before running: put your key in `.env` (see `.env.example`) and parse the corpus with `python -m src.parsing`.

In [7]:
import os
import warnings

# Keep the output clean: no progress bars and no notices while the models load.
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", message="IProgress not found")

In [8]:
import textwrap
from datetime import date
from pathlib import Path

from src.chunking import chunk_document, load_tokenizer
from src.config import load_config, project_path
from src.embedding import BgeM3Embedder
from src.env import load_env_file, require_env
from src.llm import ChatClient, Reply
from src.parsing import load_corpus
from src.retrieval import Hit, VectorIndex

config = load_config()
load_env_file(project_path(config.paths.env_file))
llm = ChatClient(config.llm, require_env(config.llm.api_key_env))


def show(reply: Reply) -> None:
    """Print an answer of the model, wrapped, with the name of the model that wrote it."""
    for paragraph in reply.text.splitlines():
        print(textwrap.fill(paragraph, width=100))
    print(f"\n[{reply.model}]")


print("model:", config.llm.model)

model: nvidia/nemotron-3-ultra-550b-a55b:free


## 1. The model alone, a general question

One call: a question goes in, text comes out.

In [9]:
show(llm.ask("In due frasi: che cos'è il recesso da un contratto?"))

Model nvidia/nemotron-3-ultra-550b-a55b:free failed with code 503: Upstream error from Nvidia: Service temporarily overloaded; trying the fallback model nvidia/nemotron-3-super-120b-a12b:free


Il recesso da un contratto è il diritto riconosciuto a una delle parti di porre fine unilateralmente
al contratto, senza necessità di consenso dell'altra parte, facendo cessare gli effetti giuridici
per il futuro. Tale diritto può essere previsto dalla legge (ad esempio, nei contratti di consumo) o
dalle clausole contrattuali e deve essere esercitato nel rispetto delle forme, dei termini e delle
eventuali condizioni stabilite.

[nvidia/nemotron-3-super-120b-a12b:free]


This is general knowledge: the model has read plenty of it during training.

The name in brackets says which model wrote the answer. Free models are shared and sometimes busy:
when a line says that the main model failed and the fallback model was tried, nothing is wrong.

## 2. The model alone, a question about us

Now something that only our documents know.

In [10]:
QUESTION = "Con quanto preavviso possiamo recedere dal contratto CTR-2024-017 con Beta Idroservizi?"

show(llm.ask(QUESTION))

Non ho accesso al testo del contratto **CTR-2024-017** né a database aziendali o registri privati.
Essendo un'intelligenza artificiale, non posso consultare documenti specifici non condivisi nella
conversazione.

Per conoscere il preavviso esatto devi **leggere la clausola di "Recesso", "Risoluzione" o "Durata e
Recesso" all'interno del contratto stesso**.

Solitamente le informazioni chiave da cercare sono:
1.  **Durata del contratto** (es. annuale, pluriennale, a tempo indeterminato).
2.  **Rinnovo automatico** (tacito rinnovo): se c'è, il preavviso va dato *prima* della scadenza per
evitare il rinnovo.
3.  **Termine per il preavviso** (es. "60 giorni prima della scadenza", "90 giorni", "6 mesi").
4.  **Modalità di comunicazione** (solitamente raccomandata A/R o PEC a un indirizzo specifico).

**Cosa puoi fare ora:**
*   Controlla la tua copia firmata del contratto (cerca parole chiave: *preavviso, recesso,
disdetta, scadenza, tacito rinnovo*).
*   Se non lo trovi, contatta il tuo **

The model says it plainly: it has no access to our contracts, and all it can offer is generic advice.
Another model, or another run, may produce a plausible number instead, which is worse.
Training does not help here: **the model needs the text**.

## 3. We paste the text by hand

We know where to look: Art. 7 of the contract is about withdrawal. We put it in the prompt, above the question.

In [11]:
documents = load_corpus(project_path(config.paths.parsed_dir))
tokenizer = load_tokenizer(config.embedding.model_name)
chunks = [chunk for document in documents for chunk in chunk_document(document, tokenizer, config.chunking)]

contract = [chunk for chunk in chunks if "01_contratti/CTR-2024-017" in chunk.source]
article_7 = next(chunk for chunk in contract if chunk.section.startswith("Art. 7"))
prompt = f"Use this text to answer.\n\n{article_7.text}\n\nQuestion: {QUESTION}"
print(prompt)

Use this text to answer.

## Art. 7 - Recesso

Il Committente può recedere dal contratto in qualsiasi momento, con preavviso di 60 (sessanta) giorni.

In caso di recesso il Committente riconosce al Fornitore, oltre al pagamento delle prestazioni eseguite, un indennizzo pari a 2 (due) mensilità del corrispettivo.

Il Fornitore può recedere dal contratto esclusivamente per giusta causa, con preavviso di 180 (centottanta) giorni, garantendo in ogni caso la continuità delle prestazioni fino al subentro di un nuovo operatore.

Question: Con quanto preavviso possiamo recedere dal contratto CTR-2024-017 con Beta Idroservizi?


In [12]:
show(llm.ask(prompt))

Model nvidia/nemotron-3-ultra-550b-a55b:free failed with code 503: Upstream error from Nvidia: Service temporarily overloaded; trying the fallback model nvidia/nemotron-3-super-120b-a12b:free


Il committente può recedere dal contratto CTR‑2024‑017 con Beta Idroservizi con un preavviso di **60
(sessanta) giorni**.

[nvidia/nemotron-3-super-120b-a12b:free]


Same model, same question: now the answer comes from our contract, **60 days** for the customer.
This is the whole idea of RAG: **the model answers from text that we put in the prompt.**

Look at the note at the end of the answer: the article alone does not say which party we are.
A passage taken out of its document loses part of its meaning.

Two things are still wrong. We had to know in advance where the answer was, and the corpus has 1754 chunks.
And we trusted one article: is it the whole story? Keep those 60 days in mind.

## 4. Retrieve: we find the text automatically

Step 2 gave us the tool. Every chunk becomes a vector, once; the question becomes a vector;
we keep the closest chunks. The vectors are saved on disk, so the slow part runs only the first time
(about four minutes on a laptop CPU).

In [13]:
embedder = BgeM3Embedder(config.embedding)
index_dir = project_path(config.paths.index_dir) / "plain"

if index_dir.exists():
    index = VectorIndex.load(index_dir)
    if index.chunks != tuple(chunks):
        raise ValueError(f"{index_dir} was built from other chunks: delete it and run this cell again")
else:
    index = VectorIndex(chunks, embedder.embed([chunk.text for chunk in chunks]).dense)
    index.save(index_dir)
print(len(index), "chunks in the index, vectors of size", index.vectors.shape[1])

1754 chunks in the index, vectors of size 1024


In [14]:
def retrieve(question: str) -> list[Hit]:
    """The chunks closest to the question; metric and number of chunks come from the configuration."""
    query = embedder.embed([question]).dense[0]
    return index.search(query, config.reranking.top_k, config.reranking.similarity_metric)


def show_hits(hits: list[Hit]) -> None:
    """Print one line per retrieved chunk: position, score, document and section."""
    for position, hit in enumerate(hits, start=1):
        document = Path(hit.chunk.source).name[:44]
        print(f"{position:2d}  {hit.score:.3f}  {document:44s}  {hit.chunk.section[:36]}")


hits = retrieve(QUESTION)
show_hits(hits)

 1  0.778  EML-2025-01_proposta-addendum-recesso-beta.t  
 2  0.764  VRB-2025-02_comitato-manutenzione_2025-01-30  2. Condizioni di recesso del contrat
 3  0.751  EML-2025-04_chiarimento-preavviso-recesso-be  
 4  0.725  EML-2025-01_proposta-addendum-recesso-beta.t  
 5  0.708  EML-2025-04_chiarimento-preavviso-recesso-be  
 6  0.696  SCH-IMP-07_scheda-impianto-pratolungo.pdf     Contratti di manutenzione e assisten
 7  0.682  SCH-IMP-04_scheda-impianto-castelrovo.pdf     Contratti di manutenzione e assisten
 8  0.659  SCH-IMP-02_scheda-impianto-pontenero.pdf      Contratti di manutenzione e assisten
 9  0.658  VRB-2025-02_comitato-manutenzione_2025-01-30  Decisioni
10  0.654  CTR-2024-017_addendum-01_Beta.pdf             premesso che


Look at what came back. The article we pasted by hand is not even in the list. Instead there are two
e-mail threads, the minutes of a committee, three plant sheets and an addendum: all of them talk about
the withdrawal from this contract. The retriever found documents that we did not know we had to read.

One of them, an e-mail of December 2025:

In [15]:
print(next(hit.chunk.text for hit in hits if "EML-2025-04" in hit.chunk.source))

Da: Marta Colombini <marta.colombini@torrevalleacque.example>
A: Luca Bassi <luca.bassi@torrevalleacque.example>
Cc: Dario Ferrante <dario.ferrante@torrevalleacque.example>
Inviato: 09/12/2025 17:40
Oggetto: R: Preavviso recesso contratto Beta

Ciao Luca,

no, attenzione. Con Beta abbiamo tre contratti e i preavvisi sono diversi.

Per il contratto di manutenzione delle pompe sommergibili, CTR-2024-017, il preavviso di recesso è di 45 giorni dal 1° marzo 2025, per effetto dell'addendum n. 1; nel testo originale del contratto era di 60 giorni.

I 30 giorni che ricordi sono quelli del contratto di pronto intervento e noleggio, CTR-2025-030. Per il contratto di fornitura delle pompe, CTR-2023-009, il preavviso è di 90 giorni.

In ogni caso il recesso si comunica solo a mezzo PEC e lo invia il mio ufficio.

Marta


The notice was changed: **45 days, not 60**, since 1 March 2025, by Addendum 1.

In section 3 we pasted the original article and got an answer that was right in 2024 and is wrong today.
One article was not the whole story.

## 5. Augment: the passages go into the prompt, with their source

The prompt has three parts: instructions that do not change, the retrieved passages, the question.
Every passage carries the name of its document, so that the model can say where a fact comes from.

In [16]:
SYSTEM = (
    "You answer questions about the documents of the company Torrevalle Acque. "
    "Use only the passages in the prompt and name the document each fact comes from. "
    "When passages disagree, explain which one applies today and why. "
    "If the passages do not contain the answer, say that you do not know. "
    f"Today is {date.today():%d %B %Y}. Answer in the language of the question."
)


def build_prompt(question: str, hits: list[Hit]) -> str:
    """The prompt of the model: the retrieved passages, numbered and with their source, then the question."""
    passages = [
        f"[{number}] {Path(hit.chunk.source).name} | {hit.chunk.section}\n{hit.chunk.text}"
        for number, hit in enumerate(hits, start=1)
    ]
    return "Passages:\n\n" + "\n\n".join(passages) + f"\n\nQuestion: {question}"


prompt = build_prompt(QUESTION, hits)
print(prompt[:900], "\n[...]\n", prompt[-90:])

Passages:

[1] EML-2025-01_proposta-addendum-recesso-beta.txt | 
Da: Roberta Gallo <roberta.gallo@betaidroservizi.example>
A: Marta Colombini <marta.colombini@torrevalleacque.example>
Cc: Dario Ferrante <dario.ferrante@torrevalleacque.example>
Inviato: 03/02/2025 09:48
Oggetto: Contratto CTR-2024-017 - proposta di addendum sul preavviso di recesso

Gentile dott.ssa Colombini,

confermiamo la disponibilità a ridurre a 45 giorni il preavviso per il recesso del Committente. Non possiamo invece accettare i 30 giorni ipotizzati in riunione: per riorganizzare le squadre ci serve più tempo.

Restano fermi il preavviso di 180 giorni per il nostro eventuale recesso e il termine di 90 giorni per la disdetta alla scadenza. Proponiamo la firma dell'addendum per lunedì 10 febbraio, con effetto dal 1° marzo.

Cordiali saluti, Roberta Gallo - Beta Idroservizi S.r.l.

[2] VRB-2025-02_comitato-manutenzio 
[...]
 n: Con quanto preavviso possiamo recedere dal contratto CTR-2024-017 con Beta Idroservizi?


## 6. Generate: the model answers from the passages

In [17]:
show(llm.ask(prompt, system=SYSTEM))

Dai documenti risulta che **il preavviso di recesso per il Committente (Torrevalle Acque) sul
contratto CTR-2024-017 è di 45 giorni**, efficace dal **1° marzo 2025** per effetto dell’addendum n.
1.

**Fonti:**
- Il verbale del comitato manutenzione del 30/01/2025 ([2], [9]) registra che il preavviso
originario era di 60 giorni, il fornitore ha accettato di ridurlo a 45 giorni (rifiutando i 30
giorni) e il comitato ha approvato la riduzione.
- L’email di Beta Idroservizi del 03/02/2025 ([1]) conferma la disponibilità a 45 giorni per il
recesso del Committente e propone la firma dell’addendum per il 10 febbraio con effetto dal 1°
marzo.
- L’email di Marta Colombini del 09/12/2025 ([3]) specifica esplicitamente: «Per il contratto di
manutenzione delle pompe sommergibili, CTR-2024-017, il preavviso di recesso è di 45 giorni dal 1°
marzo 2025, per effetto dell'addendum n. 1; nel testo originale del contratto era di 60 giorni».
- L’addendum n. 1 ([10]) premette che le parti intendono «ridurr

**45 days**, in force since 1 March 2025 because of Addendum 1. The answer lists the passages it used:
the e-mails, the minutes of the committee, the addendum. It also explains the other numbers: 60 was the
original text, 30 a proposal that was rejected.

Compare with section 3. There we chose one article and got 60 days, an answer that was right in 2024.
Here the retriever chose, and it found the documents that tell the rest of the story.

## The complete RAG

Retrieve, augment, generate: one function.

In [18]:
def answer(question: str) -> None:
    """Answer a question from the documents and list the passages that were given to the model."""
    hits = retrieve(question)
    show(llm.ask(build_prompt(question, hits), system=SYSTEM))
    print("\npassages given to the model:")
    show_hits(hits)

A question about a machine:

In [19]:
answer("Cosa significa l'allarme E-4012 sulla pompa P7?")

L'allarme **E-4012** sulla pompa **P7** indica un'**infiltrazione d'acqua nella camera olio**
(camera delle tenute).

**Dettagli dal manuale di manutenzione P7 (MM-P7 rev. 3)**
- **Gravità**: allarme; **Ripristino**: manuale.
- **Controlli**: prelevare un campione di olio dalla camera tenute; verificare lo stato del sensore
di infiltrazione (ricambio **RIC-P7-0071**).
- **Rimedio**: la pompa può restare in servizio **per non più di 24 ore**. Entro tale termine
occorre sollevarla, controllare l'olio e **sostituire la tenuta meccanica inferiore** (ricambio
**RIC-P7-0031**)【3】.

**Conferme dai documenti successivi**
- Il bollettino tecnico **BT-12** (marzo 2026) specifica che «la soglia dell'allarme E-4012 e il
relativo margine di 24 ore restano invariati»【6】.
- Nel verbale del comitato manutenzione del 30/01/2025, Ferrante ricorda che «su questo modello la
pompa può restare in servizio al massimo 24 ore dopo l'allarme»【4】.
- Due ticket di intervento su pompe P7 (TK-2025-0305 e TK-2026-01

A question about a person:

In [20]:
answer("Chi è il legale rappresentante di Beta Idroservizi?")

Il legale rappresentante di **Beta Idroservizi S.r.l.** è **Guido Santoro**.

Questa informazione risulta in tutti i contratti e addendum che coinvolgono Beta Idroservizi nei
documenti forniti:
- Contratto CTR-2023-009 (fornitura pompe sommergibili P7-P9) [1]
- Contratto CTR-2025-030 (pronto intervento e noleggio pompe) [2]
- Contratto CTR-2024-017 (manutenzione pompe sommergibili P7-P9) [3]
- Addendum n. 1 al contratto CTR-2024-017 [4]
- Addendum n. 2 al contratto CTR-2024-017 [5]

In tutti i casi la società è indicata come «Beta Idroservizi S.r.l., con sede legale in Viale delle
Officine 21, Castelrovo, codice fiscale e partita IVA 00845170992, in persona del legale
rappresentante Guido Santoro».

[nvidia/nemotron-3-ultra-550b-a55b:free]

passages given to the model:
 1  0.678  CTR-2023-009_Beta_fornitura-pompe-sommergibi  e
 2  0.678  CTR-2025-030_Beta_pronto-intervento-e-nolegg  e
 3  0.678  CTR-2024-017_Beta_manutenzione-pompe-sommerg  e
 4  0.635  CTR-2024-017_addendum-01_Beta.pd

And a question that the documents cannot answer:

In [21]:
answer("Qual è il numero di cellulare del Direttore Generale?")

Model nvidia/nemotron-3-ultra-550b-a55b:free failed with code 503: Upstream error from Nvidia: Service temporarily overloaded; trying the fallback model nvidia/nemotron-3-super-120b-a12b:free


Non ho trovato il numero di cellulare del Direttore Generale nei documenti forniti.

[nvidia/nemotron-3-super-120b-a12b:free]

passages given to the model:
 1  0.516  CTR-2024-022_addendum-01_Epsilon.pdf          ADDENDUM N. 1 AL CONTRATTO N. CTR-20
 2  0.514  CTR-2024-022_Epsilon_servizio-di-telecontrol  e
 3  0.514  CTR-2026-003_Epsilon_estensione-telecontroll  e
 4  0.513  CTR-2023-018_Omega_manutenzione-opere-civili  CONTRATTO DI MANUTENZIONE
 5  0.513  CTR-2024-011_Alfa_manutenzione-pompe-campi-p  CONTRATTO DI MANUTENZIONE
 6  0.512  CTR-2023-015_Zeta_manutenzione-elettromeccan  CONTRATTO DI MANUTENZIONE
 7  0.512  CTR-2025-008_Zeta_manutenzione-paratoie-e-gr  CONTRATTO DI MANUTENZIONE
 8  0.511  CTR-2025-024_Lambda_manutenzione-gruppi-elet  CONTRATTO DI MANUTENZIONE
 9  0.511  CTR-2024-019_Gamma_manutenzione-pompe-centri  CONTRATTO DI MANUTENZIONE
10  0.510  CTR-2025-014_Delta_manutenzione-quadri-elett  CONTRATTO DI MANUTENZIONE


## What we take away

- A model alone answers from its training: fine for general knowledge, of no use for our documents.
- RAG is three steps around the model: **retrieve** the passages, **augment** the prompt with them,
  **generate** the answer.
- Retrieval finds documents that we would not have opened: by hand we got 60 days, with retrieval 45.
- The passages reach the model with their source, so the answer can say where each fact comes from.
- The instructions say what to do when the passages do not contain the answer: the last question got
  "Non lo so", not an invented phone number. Look at its scores, all around 0.51: the retriever always
  returns ten passages, even when none is relevant.

What is still missing:

1. a way to **measure** whether the answers are right: a set of questions with the expected answer;
2. better retrieval: exact match on codes, re-ranking, the source written in the text that we embed;
3. an interface, so that asking a question does not require a notebook.